## Problem Statement

### Business Context

"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.

To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.

## Objective

As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.

## Data Description

The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:

**Customer Details**
- **CustomerID:** Unique identifier for each customer.
- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).
- **Age:** Age of the customer.
- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).
- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).
- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).
- **Gender:** Gender of the customer (Male, Female).
- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.
- **PreferredPropertyStar:** Preferred hotel rating by the customer.
- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).
- **NumberOfTrips:** Average number of trips the customer takes annually.
- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).
- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).
- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.
- **Designation:** Customer's designation in their current organization.
- **MonthlyIncome:** Gross monthly income of the customer.

**Customer Interaction Data**
- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.
- **ProductPitched:** The type of product pitched to the customer.
- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.-
- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.


## 1. Setup and data registration

In [1]:
from pathlib import Path
import json,sys,warnings
import numpy as np
import pandas as pd
from IPython.display import display,Markdown
ROOT=Path.cwd()
DATA_PATH= ROOT/"data"/"tourism.csv"
MODEL_DIR= ROOT/"models"
MODEL_DIR.mkdir(exist_ok=True)
assert DATA_PATH.exists(), f"Dataset not found: {DATA_PATH}. Run from the project root."
print("Project root:",ROOT)
print("Registered dataset:", DATA_PATH.name, f"({DATA_PATH.stat().st_size:,} bytes)")
raw = pd.read_csv(DATA_PATH)
print("Raw shape:", raw.shape)
display(raw.head())


Project root: c:\Users\ayith\OneDrive\Documents\GitHub\visit-with-us-mlops
Registered dataset: tourism.csv (477,468 bytes)
Raw shape: (4128, 21)


,Unnamed: 0,CustomerID,ProdTaken,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,...,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,0,200000,1,41.0,Self Enquiry,3,6.0,Salaried,Female,3,...,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0
1,1,200001,0,49.0,Company Invited,1,14.0,Salaried,Male,3,...,Deluxe,4.0,Divorced,2.0,0,3,1,2.0,Manager,20130.0
2,2,200002,1,37.0,Self Enquiry,1,8.0,Free Lancer,Male,3,...,Basic,3.0,Single,7.0,1,3,0,0.0,Executive,17090.0
3,3,200003,0,33.0,Company Invited,1,9.0,Salaried,Female,2,...,Basic,3.0,Divorced,2.0,1,5,1,1.0,Executive,17909.0
4,5,200005,0,32.0,Company Invited,1,8.0,Salaried,Male,3,...,Basic,3.0,Single,1.0,0,5,1,1.0,Executive,18068.0


## 2. Data preparation and quality checks

In [2]:
df=raw.copy()
df.columns=[c.strip() for c in df.columns]
df=df.loc[:, ~pd.Index(df.columns).str.match(r"^Unnamed")]
print("Rows/columns after dropping index-like columns:",df.shape)
print("Duplicate rows:",int(df.duplicated().sum()))
print("Missing values by column (non-zero only):")
missing=df.isna().sum()
display(missing[missing.gt(0)].sort_values(ascending=False).to_frame("missing_count"))
print("Target distribution:")
display(df["ProdTaken"].value_counts(dropna=False).rename(index={0:"No purchase (0)",1:"Purchase (1)"}).to_frame("count"))
display((df["ProdTaken"].value_counts(normalize=True)*100).round(2).rename("percent").to_frame())


Rows/columns after dropping index-like columns: (4128, 20)
Duplicate rows: 0
Missing values by column (non-zero only):


,missing_count


Target distribution:


,count
ProdTaken,
No purchase (0),3331
Purchase (1),797


,percent
ProdTaken,
0,80.69
1,19.31


### Observations
* Removed the csv's index-like `Unnamed: 0` column and the `CustomerID` identifier from predictors.
* Removed duplicate rows, No missing values are present.
* One-hot encoded categorical features and all learned preprocessing is inside the sklearn Pipeline to avoid train/test leakage.

## 3. Train/test split, experimentation tracking, and model selection

In [ ]:
# This is the trained file which is written in train.py
#  which pasted here for showing the code.
from pathlib import Path
import json, argparse, warnings
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, classification_report, confusion_matrix
warnings.filterwarnings("ignore")

ROOT = Path(__file__).resolve().parents[1]
def load_data(path=None):
    path = Path(path) if path else ROOT / "data" / "tourism.csv"
    df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    df = df.loc[:, ~pd.Index(df.columns).str.match(r"^Unnamed")]
    if "ProdTaken" not in df.columns:
        raise ValueError("Expected target column 'ProdTaken'.")
    df = df.drop_duplicates()
    df["ProdTaken"] = pd.to_numeric(df["ProdTaken"], errors="coerce")
    df = df.dropna(subset=["ProdTaken"])
    df["ProdTaken"] = df["ProdTaken"].astype(int)
    df = df[df["ProdTaken"].isin([0, 1])]
    # CustomerID is an identifier, not a meaningful predictive feature.
    X = df.drop(columns=["ProdTaken", "CustomerID"], errors="ignore")
    y = df["ProdTaken"]
    return df, X, y

def build_preprocessor(X):
    numeric = X.select_dtypes(include=["number"]).columns.tolist()
    categorical = X.select_dtypes(exclude=["number"]).columns.tolist()
    num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())])
    cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                         ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    return ColumnTransformer([("numeric", num_pipe, numeric), ("categorical", cat_pipe, categorical)])

def main(data_path=None, output_dir=None):
    out = Path(output_dir) if output_dir else ROOT / "models"
    out.mkdir(parents=True, exist_ok=True)
    df, X, y = load_data(data_path)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    candidates = {
        "logistic_regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42),
        "random_forest": RandomForestClassifier(n_estimators=100, min_samples_leaf=2, class_weight="balanced_subsample", random_state=42, n_jobs=2),
        "extra_trees": ExtraTreesClassifier(n_estimators=100, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=2),
    }
    results=[]; fitted={}
    for name, clf in candidates.items():
        pipe=Pipeline([("preprocess", build_preprocessor(X_train)), ("model", clf)])
        pipe.fit(X_train, y_train)
        pred=pipe.predict(X_test)
        proba=pipe.predict_proba(X_test)[:,1]
        metrics={
            "model": name, "accuracy": float(accuracy_score(y_test,pred)),
            "precision": float(precision_score(y_test,pred,zero_division=0)),
            "recall": float(recall_score(y_test,pred,zero_division=0)),
            "f1": float(f1_score(y_test,pred,zero_division=0)),
            "roc_auc": float(roc_auc_score(y_test,proba)),
            "average_precision": float(average_precision_score(y_test,proba)),
        }
        results.append(metrics); fitted[name]=pipe
    # Select by ROC-AUC; the final threshold should be chosen with business costs in mind.
    best_name=max(results,key=lambda x:x["roc_auc"])["model"]
    best=fitted[best_name]
    pred=best.predict(X_test); proba=best.predict_proba(X_test)[:,1]
    metrics_by_name={r["model"]:r for r in results}
    report={
        "dataset_rows_after_cleaning": int(len(df)),
        "feature_count": int(X.shape[1]),
        "positive_rate": float(y.mean()),
        "train_rows": int(len(X_train)), "test_rows": int(len(X_test)),
        "selected_model": best_name,
        "metrics": metrics_by_name[best_name],
        "all_model_metrics": results,
        "classification_report": classification_report(y_test,pred,output_dict=True,zero_division=0),
        "confusion_matrix": confusion_matrix(y_test,pred).tolist(),
        "feature_columns": X.columns.tolist(),
    }
    joblib.dump(best, out/"tourism_model.joblib")
    (out/"evaluation.json").write_text(json.dumps(report,indent=2))
    pd.DataFrame(results).sort_values("roc_auc",ascending=False).to_csv(out/"experiment_results.csv",index=False)
    # MLflow tracking is enabled when MLflow is installed/configured; CSV/JSON remain portable records.
    try:
        import mlflow
        mlflow.set_experiment("visit-with-us-tourism")
        for r in results:
            with mlflow.start_run(run_name=r["model"]):
                mlflow.log_params({"model":r["model"],"random_state":42,"test_size":0.2})
                mlflow.log_metrics({k:v for k,v in r.items() if k!="model"})
                mlflow.log_artifact(str(out/"experiment_results.csv"))
        report["mlflow_tracking"]="logged"
        (out/"evaluation.json").write_text(json.dumps(report,indent=2))
    except ImportError:
        report["mlflow_tracking"]="MLflow not installed in this environment; reproducible experiment metrics saved to CSV/JSON."
        (out/"evaluation.json").write_text(json.dumps(report,indent=2))
    print("Dataset rows:",len(df),"| features:",X.shape[1],"| positive rate:",round(y.mean(),4))
    print(pd.DataFrame(results).sort_values("roc_auc",ascending=False).round(4).to_string(index=False))
    print("\nSelected model:",best_name)
    print("Confusion matrix [[TN, FP], [FN, TP]]:",report["confusion_matrix"])
    print("Classification report:")
    print(classification_report(y_test,pred,zero_division=0))
    print("Saved model and evaluation artifacts to",out)
    return report

if __name__=="__main__":
    parser=argparse.ArgumentParser()
    parser.add_argument("--data",default=None)
    parser.add_argument("--output",default=None)
    args=parser.parse_args()
    main(args.data,args.output)

In [ ]:
# Run the reproducible training script which is written in train.py file.
# It evaluates Logistic Regression,
# Random Forest, and Extra Trees on a stratified 80/20 holdout split.
sys.path.insert(0, str(ROOT / "src"))
from train import main as train_and_evaluate
report = train_and_evaluate(data_path=DATA_PATH, output_dir=MODEL_DIR)
results = pd.read_csv(MODEL_DIR / "experiment_results.csv")
display(results.round(4))


Dataset rows: 4128 | features: 18 | positive rate: 0.1931
              model  accuracy  precision  recall     f1  roc_auc  average_precision
      random_forest    0.8983     0.8099  0.6164 0.7000   0.9486             0.8389
        extra_trees    0.9007     0.7421  0.7421 0.7421   0.9409             0.8267
logistic_regression    0.7324     0.3967  0.7484 0.5185   0.8195             0.5927

Selected model: random_forest
Confusion matrix [[TN, FP], [FN, TP]]: [[644, 23], [61, 98]]
Classification report:
              precision    recall  f1-score   support

           0       0.91      0.97      0.94       667
           1       0.81      0.62      0.70       159

    accuracy                           0.90       826
   macro avg       0.86      0.79      0.82       826
weighted avg       0.89      0.90      0.89       826

Saved model and evaluation artifacts to c:\Users\ayith\Downloads\Graded_Project\models


,model,accuracy,precision,recall,f1,roc_auc,average_precision
0,random_forest,0.8983,0.8099,0.6164,0.7000,0.9486,0.8389
1,extra_trees,0.9007,0.7421,0.7421,0.7421,0.9409,0.8267
2,logistic_regression,0.7324,0.3967,0.7484,0.5185,0.8195,0.5927


### Observations

- The dataset is imbalanced: purchase cases are the minority class, so accuracy alone is insufficient.
- Compared precision, recall, F1, ROC-AUC and average precision. ROC-AUC is used for model selection in this baseline.
- Candidate-model metrics are recorded in `models/experiment_results.csv` and `models/evaluation.json`. 
- MLflow tracking is enabled when MLflow is installed; the portable CSV/JSON tracking files are generated regardless.

# 4. Deployement

## Dockerfile

In [ ]:
# Pasted dockerfile which ran while running whole project at once
#  so not showing output column
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8501
CMD ["streamlit","run","app.py","--server.address=0.0.0.0","--server.port=8501"]

# Streamlit App

In [ ]:
# pasted app.py code here which ran while running whole project at a time
#  in terminal so output is not shown below
from pathlib import Path
import sys, json
import joblib
import pandas as pd
import streamlit as st

ROOT=Path(__file__).resolve().parent
MODEL_PATH=ROOT/"models"/"tourism_model.joblib"
st.set_page_config(page_title="Visit with Us | Wellness Tourism",page_icon="🌿",layout="wide")
st.title("🌿 Wellness Tourism Purchase Predictor")
st.write("Estimate the likelihood that a customer will purchase the Wellness Tourism Package. Use predictions to prioritize outreach—not to deny service.")
if not MODEL_PATH.exists():
    st.error("Model file not found. Run `python src/train.py` and commit models/tourism_model.joblib.")
    st.stop()
model=joblib.load(MODEL_PATH)
with st.form("customer_form"):
    st.subheader("Customer profile")
    c1,c2,c3=st.columns(3)
    with c1:
        age=st.number_input("Age",18,100,35)
        contact=st.selectbox("Type of contact",["Self Enquiry","Company Invited"])
        city=st.selectbox("City tier",[1,2,3],index=1)
        occupation=st.selectbox("Occupation",["Salaried","Small Business","Large Business","Free Lancer"])
        gender=st.selectbox("Gender",["Male","Female"])
        marital=st.selectbox("Marital status",["Married","Single","Divorced"])
    with c2:
        people=st.number_input("Number of people visiting",1,10,2)
        children=st.number_input("Children below age 5",0,5,0)
        stars=st.selectbox("Preferred property stars",[3,4,5],index=1)
        trips=st.number_input("Trips per year",0,30,3)
        passport=st.selectbox("Has passport?",["Yes","No"])
        car=st.selectbox("Owns car?",["Yes","No"])
    with c3:
        designation=st.selectbox("Designation",["Executive","Manager","Senior Manager","AVP","VP"])
        income=st.number_input("Monthly income",0,1000000,25000,step=1000)
        pitch=st.selectbox("Product pitched",["Basic","Deluxe","Standard","Super Deluxe","King"])
        satisfaction=st.slider("Pitch satisfaction score",1,5,3)
        followups=st.number_input("Number of follow-ups",0,10,3)
        duration=st.number_input("Pitch duration (minutes)",0.0,120.0,10.0,step=1.0)
    submitted=st.form_submit_button("Predict purchase likelihood")
if submitted:
    record=pd.DataFrame([{
        "Age":age,"TypeofContact":contact,"CityTier":city,"Occupation":occupation,
        "Gender":gender,"NumberOfPersonVisiting":people,"PreferredPropertyStar":stars,
        "MaritalStatus":marital,"NumberOfTrips":trips,"Passport":1 if passport=="Yes" else 0,
        "OwnCar":1 if car=="Yes" else 0,"NumberOfChildrenVisiting":children,
        "Designation":designation,"MonthlyIncome":income,"PitchSatisfactionScore":satisfaction,
        "ProductPitched":pitch,"NumberOfFollowups":followups,"DurationOfPitch":duration
    }])
    try:
        prob=float(model.predict_proba(record)[0,1])
        st.metric("Estimated purchase probability",f"{prob:.1%}")
        st.progress(min(max(prob,0.0),1.0))
        if prob>=0.5: st.success("Higher predicted likelihood: consider a relevant, personalized follow-up.")
        else: st.info("Lower predicted likelihood at the default 0.50 threshold. Consider gathering more information or using a low-cost campaign.")
        st.caption("This is a model estimate, not a guarantee. Validate calibration and campaign ROI before operational use.")
    except Exception as e:
        st.error(f"Prediction failed: {e}")

# Dependency Handling

In [ ]:
# pasting the requirements.txt file here
pandas>=2.0,<3.0
numpy>=1.24
scikit-learn>=1.3,<2.0
joblib>=1.3
streamlit>=1.32
mlflow>=2.12

## 4. Output evaluation

In [6]:
evaluation = json.loads((MODEL_DIR / "evaluation.json").read_text())
print("Selected model:", evaluation["selected_model"])
print("Test rows:", evaluation["test_rows"])
print("Confusion matrix [[TN, FP], [FN, TP]]:")
display(pd.DataFrame(evaluation["confusion_matrix"],
                     index=["Actual no purchase", "Actual purchase"],
                     columns=["Predicted no purchase", "Predicted purchase"]))
display(pd.DataFrame([evaluation["metrics"]]).set_index("model").round(4))
print("Classification report:")
display(pd.DataFrame(evaluation["classification_report"]).T.round(3))


Selected model: random_forest
Test rows: 826
Confusion matrix [[TN, FP], [FN, TP]]:


,Predicted no purchase,Predicted purchase
Actual no purchase,644,23
Actual purchase,61,98


,accuracy,precision,recall,f1,roc_auc,average_precision
model,,,,,,
random_forest,0.8983,0.8099,0.6164,0.7,0.9486,0.8389


Classification report:


,precision,recall,f1-score,support
0,0.913,0.966,0.939,667.000
1,0.810,0.616,0.700,159.000
accuracy,0.898,0.898,0.898,0.898
macro avg,0.862,0.791,0.819,826.000
weighted avg,0.894,0.898,0.893,826.000


**Interpretation:** the confusion matrix distinguishes false positives (unnecessary outreach) from false negatives (missed prospective buyers). Recall for the purchase class matters when the business wants to identify as many likely buyers as possible; precision matters when outreach is expensive. Select the probability threshold using campaign capacity and expected value rather than treating 0.50 as universally optimal.

## 5. Prediction smoke test

In [7]:
import joblib
model = joblib.load(MODEL_DIR / "tourism_model.joblib")
_, X_all, y_all = __import__("train").load_data(DATA_PATH)
sample = X_all.iloc[[0]].copy()
prob = float(model.predict_proba(sample)[0, 1])
print("Sample input row:")
display(sample)
print(f"Predicted purchase probability: {prob:.1%}")
print("Predicted class at default threshold 0.50:", int(prob >= 0.5))


Sample input row:


,Age,TypeofContact,CityTier,DurationOfPitch,Occupation,Gender,NumberOfPersonVisiting,NumberOfFollowups,ProductPitched,PreferredPropertyStar,MaritalStatus,NumberOfTrips,Passport,PitchSatisfactionScore,OwnCar,NumberOfChildrenVisiting,Designation,MonthlyIncome
0,41.0,Self Enquiry,3,6.0,Salaried,Female,3,3.0,Deluxe,3.0,Single,1.0,1,2,1,0.0,Manager,20993.0


Predicted purchase probability: 22.4%
Predicted class at default threshold 0.50: 0


## 6. Deployment and CI/CD

The repository includes:
- `app.py`: Streamlit frontend for entering customer details and requesting a prediction.
- `requirements.txt`: runtime dependencies.
- `Dockerfile`: container deployment option.
- `.github/workflows/pipeline.yml`: GitHub Actions training, smoke tests, artifact upload, and optional Hugging Face Space deployment.
- `tests/test_pipeline.py`: basic data and preprocessing checks.

To enable automatic deployment, create a **public Streamlit Space** on Hugging Face, then configure GitHub repository secret `HF_TOKEN` (write token) and repository variable `HF_SPACE` with value `username/space-name`. Push to `main`; the workflow trains, tests, and uploads the app/model to the Space. Never place access tokens directly in notebook/code.

## 7. Submission links

Replace these placeholders after creating your public GitHub repository and public Hugging Face Space:

- **GitHub repository:** `https://github.com/<your-username>/<your-repository>`
- **Hugging Face Space frontend:** `https://huggingface.co/spaces/<your-username>/<your-space>`

These public links cannot be truthfully supplied until the repository and Space are created under your accounts. Follow `README.md` for setup. After deployment, open the Space settings and confirm visibility is **Public**.

In [6]:
# Final artifact check
for path in [MODEL_DIR/"tourism_model.joblib", MODEL_DIR/"evaluation.json",
             MODEL_DIR/"experiment_results.csv", ROOT/"app.py",
             ROOT/".github"/"workflows"/"pipeline.yml"]:
    print(("OK   " if path.exists() else "MISSING "), path.relative_to(ROOT))
assert all((MODEL_DIR/name).exists() for name in
           ["tourism_model.joblib", "evaluation.json", "experiment_results.csv"])
print("\nNotebook workflow completed successfully.")


OK    models/tourism_model.joblib
OK    models/evaluation.json
OK    models/experiment_results.csv
OK    app.py
OK    .github/workflows/pipeline.yml

Notebook workflow completed successfully.
